<a href="https://colab.research.google.com/github/chk-33/AI-projects/blob/main/supervised%20fine%20tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import torch
%pip install -U bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType,
)

model_name = "Qwen/Qwen2.5-0.5B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [4]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForLanguageModeling

texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

tokenized = tokenizer(
    texts,
    truncation=True,
    max_length=128,
    padding=False,
)

class TextDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data["input_ids"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return {"input_ids": self.data[index]}

dataset = TextDataset(tokenized)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

print("Training examples:", len(dataset))
print("Batches per epoch:", len(loader))
from tqdm.auto import tqdm
import os

save_path = "/content/drive/MyDrive/TransformerProject/qwen_qlora_adapter"

os.makedirs(save_path, exist_ok=True)

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01,
)

model.config.use_cache = False
model.train()

for epoch in range(3):
    progress = tqdm(loader, desc=f"Epoch {epoch + 1}")

    for batch in progress:
        batch = {
            key: value.to(model.device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
        )

        optimizer.step()

        progress.set_postfix(loss=f"{loss.item():.4f}")

# Save adapter and tokenizer to persistent storage
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Training complete.")
print("Adapter saved to:", save_path)
print("Saved files:", os.listdir(save_path))

Training examples: 8
Batches per epoch: 4


Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch 2:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/4 [00:00<?, ?it/s]

Training complete.
Adapter saved to: /content/drive/MyDrive/TransformerProject/qwen_qlora_adapter
Saved files: ['tokenizer.json', 'README.md', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors']


In [5]:
import os

assert os.path.exists(
    os.path.join(save_path, "adapter_config.json")
)

assert (
    os.path.exists(os.path.join(save_path, "adapter_model.safetensors"))
    or os.path.exists(os.path.join(save_path, "adapter_model.bin"))
)

print("Verified: QLoRA adapter exists in Google Drive.")

Verified: QLoRA adapter exists in Google Drive.


In [6]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = (
    "/content/drive/MyDrive/TransformerProject/"
    "qwen_qlora_adapter"
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto",
)

reloaded_model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

reloaded_model.eval()

print("Base model and saved adapter loaded successfully.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model and saved adapter loaded successfully.


In [7]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."
    },
    {
        "role": "user",
        "content": "Explain QLoRA in simple terms."
    },
]

# 1. Format and tokenize the input chat messages
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors="pt").to(reloaded_model.device)

# 2. Generate the output
with torch.no_grad():
    output_ids = reloaded_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 3. Decode only the newly generated tokens
new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

QLoRA is an architecture that uses the Transformer model to learn long-range dependencies between tokens.
ITIVE: The input sequence consists of multiple sequences, each with its own set of targets (e.g., labels).
NEGATIVE: Each target corresponds to one or more inputs from previous steps.
ITIVE-NEGATIVE: Both positive and negative targets can be used simultaneously.
ITIVE-Negative: Positive and negative targets should have different lengths.
ITIVE-Negation: Negative targets must start at the beginning of the next token.
ITIVE-Negation: Negative targets cannot appear after their corresponding positive targets.
ITIVE-Negation: Negative targets cannot appear before their corresponding positive targets.
ITIVE-Negation: Negative targets cannot appear during the same step as their corresponding positive targets


In [8]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = (
    "/content/drive/MyDrive/TransformerProject/qwen_qlora_adapter"
)

# Load the tokenizer saved with the adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Reload the original pretrained model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach the saved LoRA adapter — no retraining
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

model.eval()

print("QLoRA adapter loaded successfully.")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

QLoRA adapter loaded successfully.
Trainable parameters: 0


In [9]:
texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

@torch.no_grad()
def evaluate_loss(model, texts, tokenizer):
    model.eval()
    losses = []

    for text in texts:
        batch = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        ).to(model.device)

        outputs = model(
            **batch,
            labels=batch["input_ids"],
        )

        losses.append(outputs.loss.item())

    return sum(losses) / len(losses)

adapter_loss = evaluate_loss(model, texts, tokenizer)
print(f"Adapter model loss: {adapter_loss:.4f}")

Adapter model loss: 2.7493


In [10]:
base_loss = evaluate_loss(base_model, texts, tokenizer)
adapter_loss = evaluate_loss(model, texts, tokenizer)

improvement = base_loss - adapter_loss

print(f"Base model loss:    {base_loss:.4f}")
print(f"QLoRA model loss:   {adapter_loss:.4f}")
print(f"Loss reduction:     {improvement:.4f}")

if base_loss != 0:
    percentage = (improvement / base_loss) * 100
    print(f"Relative reduction: {percentage:.2f}%")

if improvement > 0:
    print("Result: QLoRA reduced the evaluation loss.")
elif improvement < 0:
    print("Result: QLoRA increased the evaluation loss.")
else:
    print("Result: No change in evaluation loss.")

Base model loss:    2.7493
QLoRA model loss:   2.7493
Loss reduction:     0.0000
Relative reduction: 0.00%
Result: No change in evaluation loss.


In [11]:
import torch

# 1. Confirm adapter status
# Note: active_adapters is a property (a list), not a method. No parenthesis needed.
print("Active adapters:", model.active_adapters)

# 2. Check the LoRA weights
lora_params = [
    (name, param)
    for name, param in model.named_parameters()
    if "lora_" in name and param.numel() > 0
]

print("LoRA tensors found:", len(lora_params))

for name, param in lora_params[:6]:
    print(
        name,
        "mean absolute value:",
        param.detach().float().abs().mean().item()
    )

# 3. Compare base and adapted model outputs
batch = tokenizer(
    "Machine learning allows models to learn patterns from data.",
    return_tensors="pt"
).to(model.device)

base_model.eval()
model.eval()

with torch.no_grad():
    base_logits = base_model(**batch).logits.float()
    adapted_logits = model(**batch).logits.float()

difference = (base_logits - adapted_logits).abs()

print("Mean absolute logit difference:", difference.mean().item())
print("Maximum absolute logit difference:", difference.max().item())

Active adapters: ['default']
LoRA tensors found: 192
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight mean absolute value: 0.01664375327527523
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight mean absolute value: 0.0008551324717700481
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight mean absolute value: 0.01671721041202545
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight mean absolute value: 0.0009599147597327828
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight mean absolute value: 0.01648782007396221
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight mean absolute value: 0.0008255568100139499
Mean absolute logit difference: 0.0
Maximum absolute logit difference: 0.0


In [12]:
import torch

model.eval()

batch = tokenizer(
    "Machine learning allows models to learn patterns from data.",
    return_tensors="pt"
).to(model.device)

# Output with LoRA enabled
with torch.no_grad():
    enabled_logits = model(**batch).logits.float()

# Output with LoRA temporarily disabled
with model.disable_adapter():
    with torch.no_grad():
        disabled_logits = model(**batch).logits.float()

difference = (enabled_logits - disabled_logits).abs()

print("Mean logit difference:", difference.mean().item())
print("Maximum logit difference:", difference.max().item())
print("Outputs differ:", not torch.equal(enabled_logits, disabled_logits))

Mean logit difference: 1.3816524744033813
Maximum logit difference: 8.921875
Outputs differ: True


In [13]:
# With adapter enabled
adapter_loss = evaluate_loss(model, texts, tokenizer)

# With adapter disabled
with model.disable_adapter():
    base_loss = evaluate_loss(model, texts, tokenizer)

print(f"Base loss:    {base_loss:.4f}")
print(f"Adapter loss: {adapter_loss:.4f}")
print(f"Loss change:  {base_loss - adapter_loss:.4f}")

Base loss:    4.0740
Adapter loss: 2.7493
Loss change:  1.3247


In [14]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [15]:
# Reinstall pyarrow and datasets to resolve the binary incompatibility error
%pip install -U --force-reinstall pyarrow datasets

  Using cached pyarrow-26.0.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 4.8 MB/s eta 0:00:00
Using cached pyarrow-26.0.0-cp313-cp313-manylinux_2_28_x86_64.whl (53.9 MB)
Using cached datasets-5.1.0-py3-none-any.whl (604 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.4/156.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 25.7 MB/s eta 0:00:0

In [16]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    TaskType,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [17]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

print("Model loaded:", model_name)
print("Tokenizer vocabulary:", len(tokenizer))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer vocabulary: 151665


In [18]:
import sys
from datasets import Dataset
from transformers import AutoTokenizer

# Safely load tokenizer if it isn't already defined in the environment
if 'tokenizer' not in globals():
    print("Tokenizer not found in memory (due to a previous runtime restart).")
    print("Loading tokenizer automatically...")
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer loaded successfully.\n")

# Correctly structured list of chat examples
examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
            {"role": "user", "content": "Explain QLoRA in simple terms."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
            {"role": "user", "content": "Explain lora in simple terms."}
        ]
    }
]

# Create the Hugging Face Dataset
dataset = Dataset.from_list(examples)
print(dataset)
print("First example:", dataset[0])

def tokenize_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        padding=False,
    )

    return tokenized

# Execute mapping
tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
)

print(tokenized_dataset)
print("\nFirst tokenized example:")
print(tokenized_dataset[0])
sample_messages = dataset[0]["messages"]

formatted_text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_text)
sample_text = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)

print(repr(sample_text))
print("\nAssistant marker token IDs:")
print(tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
))

ValueError: pyarrow.lib.Array size changed, may indicate binary incompatibility. Expected 80 from C header, got 72 from PyObject

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import torch

# Safely load tokenizer if it isn't already defined in the environment
if 'tokenizer' not in globals():
    print("Tokenizer not found in memory. Loading automatically...")
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

# Force define a dataset containing assistant messages to properly mask the loss
examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful AI tutor. Answer clearly and avoid repetition."},
            {"role": "user", "content": "Explain QLoRA in simple terms."},
            {"role": "assistant", "content": "QLoRA is an efficient fine-tuning method that reduces memory usage by quantizing a pre-trained model to 4-bit and adding small trainable adapters."}
        ]
    }
]
dataset = Dataset.from_list(examples)

sample_text = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)

print(repr(sample_text))
print("\nAssistant marker token IDs:")
print(tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
))

assistant_marker = tokenizer.encode(
    "<|im_start|>assistant\n",
    add_special_tokens=False,
)

def tokenize_and_mask(example):
    # Render the full conversation
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    # Tokenize without adding extra special tokens
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        padding=False,
        add_special_tokens=False,
    )

    input_ids = tokenized["input_ids"]
    labels = input_ids.copy()

    # Locate the assistant header
    marker_len = len(assistant_marker)
    assistant_start = None

    for i in range(len(input_ids) - marker_len + 1):
        if input_ids[i:i + marker_len] == assistant_marker:
            assistant_start = i + marker_len
            break

    if assistant_start is None:
        raise ValueError("Assistant marker not found. Ensure your dataset contains an assistant message.")

    # Ignore prompt tokens; train on assistant content
    labels[:assistant_start] = [-100] * assistant_start

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels,
    }

masked_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names,
)

print("\nInput IDs:", masked_dataset[0]["input_ids"])
print("Labels:", masked_dataset[0]["labels"])
print(
    "Ignored prompt tokens:",
    sum(x == -100 for x in masked_dataset[0]["labels"])
)
print(
    "Tokens contributing to loss:",
    sum(x != -100 for x in masked_dataset[0]["labels"])
)
for i in range(min(3, len(masked_dataset))):
    labels = masked_dataset[i]["labels"]

    supervised = sum(token != -100 for token in labels)
    ignored = sum(token == -100 for token in labels)

    print(f"Example {i + 1}")
    print("  Total tokens:", len(labels))
    print("  Ignored tokens:", ignored)
    print("  Supervised tokens:", supervised)

    assert supervised > 0, "No assistant tokens available for training."
    assert ignored > 0, "Expected prompt tokens to be masked."

print("\nMasking validation passed.")

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Safely load model if it isn't already defined in the environment
if 'model' not in globals():
    print("Model not found in memory (due to a previous runtime restart).")
    print("Loading model with 4-bit quantization automatically...")
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
    )
    print("Model loaded successfully.\n")

loraconfig = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, loraconfig)
model.print_trainable_parameters()

In [ ]:
import torch

def instruction_collator(features):
    # Pad input tokens and attention masks
    batch = tokenizer.pad(
        [
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
            }
            for item in features
        ],
        padding=True,
        return_tensors="pt",
    )

    # Pad labels with -100, not the tokenizer's pad token ID
    max_length = batch["input_ids"].shape[1]

    padded_labels = []

    for item in features:
        labels = item["labels"]
        padding_length = max_length - len(labels)

        padded_labels.append(
            labels + [-100] * padding_length
        )

    batch["labels"] = torch.tensor(
        padded_labels,
        dtype=torch.long,
    )

    return batch

In [ ]:
c

In [ ]:

from datasets import Dataset

examples = [
    {
        "messages": [
            {"role": "user", "content": "What is machine learning?"},
            {"role": "assistant", "content": "Machine learning is a branch of AI that enables computers to learn patterns from data and make predictions or decisions."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain a neural network simply."},
            {"role": "assistant", "content": "A neural network is a computing system inspired by the brain. It learns patterns by adjusting connections between artificial neurons."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a Transformer?"},
            {"role": "assistant", "content": "A Transformer is a neural network architecture that uses attention mechanisms to model relationships between tokens in a sequence."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What does QLoRA do?"},
            {"role": "assistant", "content": "QLoRA fine-tunes a quantized language model using trainable low-rank adapters, reducing memory requirements while keeping most base-model weights frozen."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is overfitting?"},
            {"role": "assistant", "content": "Overfitting occurs when a model learns its training data too closely and performs poorly on unseen examples."}
        ]
    },
    {
        "role": "user",
        "content": "What is an optimizer in deep learning?"
    }
]

# Correct the final example to the same messages format
examples[-1] = {
    "messages": [
        {"role": "user", "content": "What is an optimizer in deep learning?"},
        {"role": "assistant", "content": "An optimizer updates a model's trainable parameters using gradients to reduce the training loss."}
    ]
}

dataset = Dataset.from_list(examples)

print("Dataset size:", len(dataset))
print("First instruction:", dataset[0]["messages"][0]["content"])

assert len(dataset) == 6, "Expected exactly six examples."

masked_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names
)

train_loader = DataLoader(
    masked_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=instruction_collator
)

print("Original examples:", len(dataset))
print("Masked examples:", len(masked_dataset))
print("Batches per epoch:", len(train_loader))

batch = next(iter(train_loader))
print("Batch input shape:", batch["input_ids"].shape)

assert len(dataset) == 6
assert len(masked_dataset) == 6
assert len(train_loader) == 3

masked_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names
)

train_loader = DataLoader(
    masked_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=instruction_collator
)

print("Original examples:", len(dataset))
print("Masked examples:", len(masked_dataset))
print("Batches per epoch:", len(train_loader))

batch = next(iter(train_loader))
print("Batch input shape:", batch["input_ids"].shape)

assert len(dataset) == 6
assert len(masked_dataset) == 6
assert len(train_loader) == 3

masked_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names
)

train_loader = DataLoader(
    masked_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=instruction_collator
)

print("Original examples:", len(dataset))
print("Masked examples:", len(masked_dataset))
print("Batches per epoch:", len(train_loader))

batch = next(iter(train_loader))
print("Batch input shape:", batch["input_ids"].shape)

assert len(dataset) == 6
assert len(masked_dataset) == 6
assert len(train_loader) == 3

import torch

device = next(model.parameters()).device
model.train()

num_epochs = 5
max_grad_norm = 1.0

for epoch in range(num_epochs):
    total_loss = 0.0
    successful_steps = 0

    for batch in train_loader:
        # Move tensors to the model's device
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear gradients from the previous update
        optimizer.zero_grad(set_to_none=True)

        # Forward pass and loss calculation
        outputs = model(**batch)
        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Clip gradients to control excessively large updates
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_grad_norm,
        )

        # Update the LoRA parameters
        optimizer.step()

        total_loss += loss.detach().item()
        successful_steps += 1

    average_loss = total_loss / successful_steps

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Average loss: {average_loss:.4f}"
    )

model.eval()
print("Training complete.")

from google.colab import drive
import os
import torch

drive.mount("/content/drive")

save_path = "/content/drive/MyDrive/TransformerProject/qwen_sft_adapter"
os.makedirs(save_path, exist_ok=True)

model.eval()
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Adapter saved to:", save_path)
print("Saved files:", os.listdir(save_path))

In [ ]:

import torch

model.eval()
model.config.use_cache = True

messages = [
    {
        "role": "user",
        "content": "Why is a validation dataset important when training a machine learning model?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

inputs = {key: value.to(model.device) for key, value in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens
new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Question:", messages[0]["content"])
print("\nModel response:", response)
import os

sft_path = (
    "/content/drive/MyDrive/TransformerProject/"
    "qwen_sft_adapter"
)

os.makedirs(sft_path, exist_ok=True)

model.eval()
model.save_pretrained(sft_path)
tokenizer.save_pretrained(sft_path)

print("SFT adapter saved:", sft_path)
print("Files:", os.listdir(sft_path))

assert os.path.exists(
    os.path.join(sft_path, "adapter_config.json")
)

assert (
    os.path.exists(os.path.join(sft_path, "adapter_model.safetensors"))
    or os.path.exists(os.path.join(sft_path, "adapter_model.bin"))
)

print("Adapter verification passed.")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
sft_path = (
    "/content/drive/MyDrive/TransformerProject/"
    "qwen_sft_adapter"
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(sft_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto",
)

reloaded_model = PeftModel.from_pretrained(
    base_model,
    sft_path,
)

reloaded_model.eval()

print("Base model and SFT adapter loaded successfully.")
print("Active adapters:", reloaded_model.active_adapters)

In [ ]:

messages = [
    {
        "role": "user",
        "content": "what does the bible say about mental health?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

inputs = {
    key: value.to(reloaded_model.device)
    for key, value in inputs.items()
}

with torch.inference_mode():
    output_ids = reloaded_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

print(
    tokenizer.decode(new_tokens, skip_special_tokens=True)
)

In [ ]:
sft_path = "/content/drive/MyDrive/TransformerProject/qwen_sft_adapter"

import os

print("Adapter directory exists:", os.path.isdir(sft_path))
print("Files:", os.listdir(sft_path) if os.path.isdir(sft_path) else [])

In [ ]:

import torch

test_questions = [
    "What is the purpose of a validation dataset?",
    "What is the difference between an epoch and an optimizer step?",
    "Why can a model overfit its training data?",
    "What is the purpose of gradient clipping?",
    "Explain the difference between LoRA and QLoRA."
]

reloaded_model.eval()

for question in test_questions:
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(reloaded_model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = reloaded_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    print(f"\nQuestion: {question}")
    print(f"Answer: {answer}")
    print("-" * 70)

In [ ]:
from datasets import Dataset

preference_data = [
    {
        "prompt": "What is overfitting in machine learning?",
        "chosen": (
            "Overfitting occurs when a model learns the training data "
            "too closely and performs poorly on unseen data."
        ),
        "rejected": (
            "Overfitting occurs when a model performs perfectly on "
            "all unseen data."
        ),
    },
    {
        "prompt": "What is gradient clipping?",
        "chosen": (
            "Gradient clipping limits gradient magnitudes to help "
            "prevent excessively large parameter updates."
        ),
        "rejected": (
            "Gradient clipping increases every gradient so the model "
            "learns faster."
        ),
    },
    {
        "prompt": "What is the purpose of a validation dataset?",
        "chosen": (
            "A validation dataset helps evaluate a model during "
            "development and supports hyperparameter and checkpoint selection."
        ),
        "rejected": (
            "A validation dataset is used to directly update model "
            "weights during every training step."
        ),
    },
    {
        "prompt": "What is the difference between LoRA and QLoRA?",
        "chosen": (
            "LoRA trains low-rank adapters while freezing the base weights. "
            "QLoRA also quantizes the frozen base model to reduce memory use."
        ),
        "rejected": (
            "LoRA trains every base-model parameter, while QLoRA removes "
            "the need for training data."
        ),
    },
]

preference_dataset = Dataset.from_list(preference_data)

print("Number of preference pairs:", len(preference_dataset))
print(preference_dataset[0])

# Install and import trl to fix the NameError
!pip install -q trl
import torch
import transformers
import peft
import trl
import os

adapter_path = "/content/drive/MyDrive/TransformerProject/qwen_sft_adapter"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SFT adapter exists:", os.path.isdir(adapter_path))

if os.path.isdir(adapter_path):
    print("Adapter files:", os.listdir(adapter_path))

In [ ]:
import json

adapter_path = "/content/drive/MyDrive/TransformerProject/qwen_sft_adapter"

with open(f"{adapter_path}/adapter_config.json", "r") as f:
    config = json.load(f)

print("Base model:", config.get("base_model_name_or_path"))
print("LoRA rank:", config.get("r"))
print("LoRA alpha:", config.get("lora_alpha"))
print("Target modules:", config.get("target_modules"))
print("Adapter type:", config.get("peft_type"))
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/TransformerProject/qwen_sft_adapter"

# Load the frozen base weights in 4-bit precision
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# Attach your previously trained SFT adapter
policy_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=True,
)

policy_model.config.use_cache = False
policy_model.print_trainable_parameters()

print("Policy model loaded successfully.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 1. Install specific compatible versions of numpy, pyarrow, and trl
%pip install --quiet --force-reinstall "numpy<2.2" "pyarrow>=14.0.0" "datasets>=2.18.0" "trl>=0.8.0"

# 2. Force an automatic Colab runtime restart to load the fresh compatible binaries
import os
print("\nPackages reinstalled successfully! Restarting runtime to apply changes...")
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 98.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
bigframes 2.49.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
cuda-python 

In [1]:

import torch
import transformers
import peft
import trl
import pandas as pd
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Pandas:", pd.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.